<div style="text-align:right">
<img src="imgs/logo-curso.jpg" width="30%" style="max-width:100%; padding-right:20px;">
</div>

**Fundamentos de la Inteligencia Artificial y Programación de APIs para LLMs**

# **04. Aprendizaje de la neurona**

Ya tenemos a nuestra neurona con una función de activación continua y derivable. Ahora vamos a enseñarle a aprender. Para ello, vamos a utilizar un procedimiento llamado **entrenamiento**: una técnica que ajusta, poco a poco, los pesos de la neurona para que se acerque a la salida deseada. La idea del entrenamiento es empezar con unos pesos aleatorios al principio, observar las respuestas, calcular un error y ajustar los pesos para intentar reducirlo. Repetimos ese ciclo muchas veces. Para ello tenemos que utilizar un conjunto de ejemplos de entrenamiento.

## **1. Conjunto de datos de entrenamiento**

El conjunto de datos de entrenamiento es un conjunto de ejemplos que la neurona utilizará para aprender. Cada ejemplo consiste en un par de características (entradas) y una etiqueta que determina la clase o categoría a la que pertenece ese ejemplo. En nuestro caso, vamos a utilizar un conjunto de datos muy simple con solo tres puntos: dos azules y uno rojo (clases 1 y 0). La ventaja de este ejemplo es que podemos mirar los datos, calcular respuestas e interpretar la frontera que aprende.


|  e₁ |  e₂ | Etiqueta |
|---|---|---|
| 1 | 2 | 1 |
| 3 | 1 | 1 |
| 4 | 5 | 0 |

Las etiquetas 1 y 0 son las respuestas que queremos reproducir. No se deducen del color automáticamente: el color es una forma de dibujar las etiquetas. Los dos azules deben terminar con una salida próxima a 1 y el rojo con una salida próxima a 0.

<img src="imgs/points.png" alt="Los tres ejemplos del entrenamiento: A y B con etiqueta 1; C con etiqueta 0." width="400" style="max-width:100%;height:auto;display:block;margin-left:auto;margin-right:auto;">


El objetivo es encontrar los pesos  $w_0$, $w_1$ y $w_2$ de una neuronal artificial para que esta pueda clasificar las muestras correctamente. Este proceso va a ser automático e iterativo. Al principio, la neurona se va a inicializar con valores totalmente aleatorios para los pesos y, posteriormente, se verá qué error ha cometido en la clasificación.

In [1]:
import numpy as np

# Recuerda que la primera columna con unos corresponde a la entrada e0 que siempre es 1.
x_data = [[1, 1, 2],
          [1, 3, 1],
          [1, 4, 5]]

x_data = np.matrix(x_data)

y_data = [1, 1, 0]  # etiquetas de clase para cada ejemplo

weights = np.random.uniform(low=-1, high=1, size=3).reshape((3,1))  # Pesos aleatorios iniciales

print("Pesos aleatorios iniciales")
for i in range(len(weights)):
    print(f"w{i}: {weights[i,0]:.4f}")

def sigmoid(x):
    return 1.0/(1.0 + np.exp(-x))

output = sigmoid(x_data * weights)

output = np.asarray(output).flatten().tolist() # Convertimos a lista para poder iterar

print("\nResultado de la clasificación")
for i in range(len(output)):
    print(f"Ejemplo {i+1}: Salida: {output[i]:.4f}, Etiqueta: {y_data[i]}")


Pesos aleatorios iniciales
w0: -0.2400
w1: 0.9438
w2: -0.2372

Resultado de la clasificación
Ejemplo 1: Salida: 0.5571, Etiqueta: 1
Ejemplo 2: Salida: 0.9133, Etiqueta: 1
Ejemplo 3: Salida: 0.9129, Etiqueta: 0


En este momento inicial, la neurona no sabe nada. No tiene información sobre los datos, ni sobre las etiquetas. Por eso, al principio, sus respuestas van a ser muy probablemente incorrectas. Pero poco a poco, con el entrenamiento, la neurona va a ir ajustando sus pesos para acercarse a las respuestas correctas. Vamos a verlo.

## **2. Función de error: cuánto nos equivocamos**

Lo primero que necesitamos es una forma de medir el error. Para ello vamos a comparar la salida de la neurona con la etiqueta. La función de error nos dirá cuánto nos hemos equivocado. Si la salida de la neurona es 0.8 y la etiqueta es 1, el error es 0.2; si la salida es 0.1 y la etiqueta es 1, el error es 0.9. Vamos a sumar los errores de todos los ejemplos para obtener un error total. Para ello, la fórmula que vamos a utilizar es la **suma de los cuadrados de los errores**.

$$
\text{error total} = J(\vec{w})= \sum_{j=1}^{3} \left[\sigma \left( \sum_{i=0}^{2} w_i e_{i}^j \right) - l^j \right]^2
$$

$j$ es el índice de cada ejemplo del conjunto de entrenamiento, $i$ es el índice de cada característica de cada ejemplo, $w_i$ es el peso de la característica $i$, $e_{i}^j$ es la característica $i$ del ejemplo $j$, $\sigma$ es la función de activación sigmoide y $l^j$ es la etiqueta del ejemplo $j$.

Por tanto, el error cometido en el código anterior será:

In [2]:
for out, y in zip(output, y_data):
    print(f"Salida: {out:.4f}, Etiqueta: {y}, Diferencia: {out-y:.4f}, Diferencia al cuadrado: {(out-y)**2:.4f}")

error = 0.0
for out, y in zip(output, y_data):
    error += (out-y)**2  

print(f"\nError cuadrático total: {error:.4f}")
    
    

Salida: 0.5571, Etiqueta: 1, Diferencia: -0.4429, Diferencia al cuadrado: 0.1961
Salida: 0.9133, Etiqueta: 1, Diferencia: -0.0867, Diferencia al cuadrado: 0.0075
Salida: 0.9129, Etiqueta: 0, Diferencia: 0.9129, Diferencia al cuadrado: 0.8334

Error cuadrático total: 1.0370


Es muy importante entender ahora que esta función de error está en función de los pesos, no de los ejemplos. Los ejemplos son estáticos, los pesos, no. Los pesos los iremos variando a medida que descendamos por el gradiente. Vemos también que esta función de error es continua y derivable en todo momento. Por eso nos interesaba prescindir de la función escalón $x \geq 0$ y quedarmos con una función como la sigmoide, que es continua y derivable.

## **3. Descendiendo por el gradiente**

Esta **función de error** decrece cuando las salidas de la neurona se acercan a las etiquetas (*labels*) de cada ejemplo. Por lo tanto, iremos descenciendo por el gradiente de la función de error hasta intentar alcanzar el mínimo de esta función. Recordamos que el desceso requiere el cálculo de las derivadas parciales de la función. Por tanto, como veíamos en el notebook anterior, desdeceder por el gradiente será:

$$
\vec{w} \leftarrow \vec{w} - \gamma \nabla J(\vec{w})
$$

O, expresado en términos de cada peso:

$$
w_0 \leftarrow w_0 - \gamma \frac{\partial J(\vec{w})}{\partial w_0}$$
$$
w_1 \leftarrow w_1 - \gamma \frac{\partial J(\vec{w})}{\partial w_1}$$
$$
w_2 \leftarrow w_2 - \gamma \frac{\partial J(\vec{w})}{\partial w_2}
$$  


Aquí calculamos la contribución de un solo ejemplo. En el entrenamiento usaremos los tres antes de actualizar. Así todas las contribuciones se calculan con los mismos pesos actuales.

## **4. Código: el proceso completo**

Veamos todo el proceso completo en código. Programaremos la función sigmoide y su derivada e iteraremos un determinado número de veces descendiendo por el gradiente de la función de error.

In [3]:
import numpy as np

data = [[1, 2],
          [3, 1],
          [4, 5]]

labels = [1, 1, 0]

def sigmoid(x):
    return 1.0/(1.0 + np.exp(-x))


def sigmoid_derivate(o):
    return o * (1.0 - o)  # Recordemos que si o = sigmoid(x) entonces su derivada es o*(1-o)


def train(x_data, y_data):

    w0, w1, w2 = np.random.rand(3)  # Pesos aleatorios iniciales
    gamma = 0.1 # Tasa de aprendizaje
    epochs = 10000 # Número de iteraciones

    print("Training...") # Comenzamos el entrenamiento

    for _ in range(epochs):
        
        w0_derivada = [] # Almacenaremos las derivadas parciales de la función de error con respecto a cada peso y cada ejemplo
        w1_derivada = []
        w2_derivada = []
        
        for data, label in zip(x_data, y_data): # Recorremos cada ejemplo y su etiqueta correspondiente
            
            e1, e2 = data; # Las entradas de cada ejemplo
            e0 = 1.0 # La entrada e0 siempre es 1.0
            out = sigmoid(w0*e0 + w1*e1 + w2*e2)
            aux = 2.*(out - label) * sigmoid_derivate(out) # Esta es parte de la derivada parcial de la función de error con respecto a los pesos. La otra parte es la entrada correspondiente.

            w0_derivada.append(aux * 1.0)
            w1_derivada.append(aux * e1)
            w2_derivada.append(aux * e2)
            
        w0 = w0 - gamma * np.sum(w0_derivada) # Sumamos todas las derivadas parciales de cada ejemplo y actualizamos el valor del peso correspondiente
        w1 = w1 - gamma * np.sum(w1_derivada)
        w2 = w2 - gamma * np.sum(w2_derivada)

    # Una vez entrenada la neurona, vamos a ver cómo clasifica los ejemplos de entrenamiento

    for data, label in zip(x_data, y_data):
        e1, e2 = data
        out = sigmoid(w0*1.0 + w1*e1 + w2*e2)
        print(data, "-> etiqueta:", label, "predicción:", round(out, 3))
        print("------------------------------------------")
    
    print("Pesos: ", w0, w1, w2)


train(data, labels)


Training...
[1, 2] -> etiqueta: 1 predicción: 0.991
------------------------------------------
[3, 1] -> etiqueta: 1 predicción: 0.995
------------------------------------------
[4, 5] -> etiqueta: 0 predicción: 1.0
------------------------------------------
Pesos:  1.5887446770061209 0.8850809147544673 1.098420808304778


Finalmente, vemos cómo la neurona ha ajustado sus pesos hasta llevarlos a unos valores que hacen que sus resultados (predicciones) se aproximen mucho a las etiquetas.

## **5. De los pesos a la recta**

Vamos a hacer el proceso inverso: a partir de los pesos que hemos aprendido, vamos a calcular la pendiente y término independiente de la recta que separa las dos clases. Si tomamos los pesos ($w_0 \approx 8, w_1 \approx -0.6, w_2 \approx -2$) y los representamos como la típica recta a la que estamos acostumbrados tendremos:

$$ 8e_0 -0.6e_1 -2e_2 = 0 $$

cambiamos los nombres ($e_1 = x, e_2=y$, recuerda que $e_0=1$)...

$$ 8 -0.6x -2y = 0 $$

y despejamos la $y$.

$$ y = -0.3x + 4 $$

Y si representamos la ecuación gráficamente

<div style="text-align: center;">
    <img src="imgs/recta.png" width="30%">
</div>

Por tanto, la neurona es capaz de clasificar todos los puntos (ejemplos) correctamente. Realmente, la neurona no va a devolver un $1$ si la etiqueta es un $1$, pero sí un valor muy cercano. Así, si la neurona devuelve un valor mayor que $0.5$ decimos que el punto tiene la etiqueta $1$. Y la etiquetaría como $0$ en caso contrario.

Lo que acabamos de ver se conoce como **aprendizaje supervisado**. Consiste en que un modelo, en este caso una neurona, se adapta (aprende sus pesos) para clasificar un conjunto de datos etiquetados. Ahora, si aparecieran nuevos puntos no etiquetados, la neurona sabría cómo clasificarlos correctamente.

## **6. Demostrador de neurona artificial**

Este [demostrador original](https://cayetanoguerra.github.io/ia/sim/canvas.html) permite complementar el estudio visualmente. Prueba a añadir unos puntos para dos clases y a entrenar la neurona. Verás cómo ajusta sus pesos para clasificar correctamente los puntos.

[← Notebook 3](03-descenso-por-el-gradiente.ipynb) · [Notebook 5 →](05-perceptron-multicapa.ipynb)